In [1]:
import os

# Clean up any previous attempts and clone the official D-FINE repository
!rm -rf D-FINE
!git clone https://github.com/Peterande/D-FINE

# Check if the D-FINE directory was created
if os.path.exists('D-FINE'):
    %cd D-FINE
    # Install dependencies
    !pip install -r requirements.txt
    !pip install -U lycoris-lora # Often needed for specific fine-tuning tasks
else:
    print("Error: D-FINE repository could not be cloned. Please check your network connection or the repository URL and try again.")

Cloning into 'D-FINE'...
remote: Enumerating objects: 1401, done.
remote: Counting objects: 100% (664/664), done.
remote: Compressing objects: 100% (286/286), done.
remote: Total 1401 (delta 542), reused 378 (delta 378), pack-reused 737 (from 3)
Receiving objects: 100% (1401/1401), 468.26 KiB | 23.41 MiB/s, done.
Resolving deltas: 100% (893/893), done.
/content/D-FINE
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 585.6/585.6 kB 53.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.6/61.6 kB 8.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 76.3/76.3 kB 8.3 MB/s eta 0:00:00


In [ ]:

# --- Patch D-FINE to export COCO-format predictions during --test-only runs ---
import os, re

dfine_path = "/content/D-FINE"
assert os.path.isdir(dfine_path), f"D-FINE not found at {dfine_path}"

def find_file_containing(pattern: str):
    hits = []
    for root, _, files in os.walk(dfine_path):
        for fn in files:
            if not fn.endswith(".py"):
                continue
            fp = os.path.join(root, fn)
            try:
                txt = open(fp, "r", encoding="utf-8").read()
            except Exception:
                continue
            if pattern in txt:
                hits.append(fp)
    return hits

coco_files = find_file_containing("class CocoEvaluator")
if not coco_files:
    raise FileNotFoundError("Could not find 'class CocoEvaluator' in the D-FINE repo.")

def score_fp(fp: str) -> int:
    txt = open(fp, "r", encoding="utf-8").read()
    return int("def update" in txt) + int("def accumulate" in txt) + int("coco_eval" in txt)

coco_files = sorted(coco_files, key=score_fp, reverse=True)
coco_fp = coco_files[0]
print("✅ CocoEvaluator file:", coco_fp)

txt = open(coco_fp, "r", encoding="utf-8").read()
if "DFINE_PRED_EXPORT_PATCH" in txt:
    print("No patch applied (already patched).")
else:
    lines = txt.splitlines(True)
    out = []
    in_init = in_update = in_acc = False
    init_inserted = update_inserted = acc_inserted = False

    for line in lines:
        out.append(line)

        # __init__
        if re.match(r"\s*def __init__\(", line):
            in_init = True
            continue
        if in_init and (not init_inserted) and re.match(r"^\s{4,}\S", line):
            out.append("        # DFINE_PRED_EXPORT_PATCH: buffer for COCO-format predictions\n")
            out.append("        self._pred_buffer = []\n")
            out.append("        self._pred_dumped = False\n")
            init_inserted = True
        if in_init and re.match(r"^\s*def\s+", line) and not re.match(r"\s*def __init__", line):
            in_init = False

        # update
        if re.match(r"\s*def update\(", line):
            in_update = True
            continue
        if in_update and (not update_inserted) and re.match(r"^\s{4,}return\b", line):
            # insert before return
            out.pop()  # remove return line
            out.append("        # DFINE_PRED_EXPORT_PATCH: export predictions if DFINE_PRED_PATH is set\n")
            out.append("        pred_path = os.getenv('DFINE_PRED_PATH')\n")
            out.append("        if pred_path:\n")
            out.append("            try:\n")
            out.append("                import torch\n")
            out.append("                for image_id, output in predictions.items():\n")
            out.append("                    if output is None:\n")
            out.append("                        continue\n")
            out.append("                    boxes = output.get('boxes', None)\n")
            out.append("                    scores = output.get('scores', None)\n")
            out.append("                    labels = output.get('labels', None)\n")
            out.append("                    if boxes is None or scores is None or labels is None:\n")
            out.append("                        continue\n")
            out.append("                    if isinstance(boxes, torch.Tensor): boxes = boxes.detach().cpu().tolist()\n")
            out.append("                    if isinstance(scores, torch.Tensor): scores = scores.detach().cpu().tolist()\n")
            out.append("                    if isinstance(labels, torch.Tensor): labels = labels.detach().cpu().tolist()\n")
            out.append("                    for (x1,y1,x2,y2), s, lab in zip(boxes, scores, labels):\n")
            out.append("                        self._pred_buffer.append({\n")
            out.append("                            'image_id': int(image_id),\n")
            out.append("                            'category_id': int(lab),\n")
            out.append("                            'bbox': [float(x1), float(y1), float(x2-x1), float(y2-y1)],\n")
            out.append("                            'score': float(s),\n")
            out.append("                        })\n")
            out.append("            except Exception as e:\n")
            out.append("                print('⚠️ Prediction export failed in update():', e)\n")
            out.append(line)  # re-add return
            update_inserted = True
        if in_update and re.match(r"^\s*def\s+", line) and not re.match(r"\s*def update\(", line):
            in_update = False

        # accumulate
        if re.match(r"\s*def accumulate\(", line):
            in_acc = True
            continue
        if in_acc and (not acc_inserted) and re.match(r"^\s{4,}\S", line):
            out.append("        # DFINE_PRED_EXPORT_PATCH: dump predictions once at accumulate()\n")
            out.append("        pred_path = os.getenv('DFINE_PRED_PATH')\n")
            out.append("        if pred_path and (not getattr(self, '_pred_dumped', False)):\n")
            out.append("            try:\n")
            out.append("                import json, os\n")
            out.append("                os.makedirs(os.path.dirname(pred_path), exist_ok=True)\n")
            out.append("                with open(pred_path, 'w') as f:\n")
            out.append("                    json.dump(getattr(self, '_pred_buffer', []), f)\n")
            out.append("                self._pred_dumped = True\n")
            out.append("                print(f\"✅ Saved predictions to {pred_path} ({len(getattr(self, '_pred_buffer', []))} dets)\")\n")
            out.append("            except Exception as e:\n")
            out.append("                print('⚠️ Prediction export failed in accumulate():', e)\n")
            acc_inserted = True
        if in_acc and re.match(r"^\s*def\s+", line) and not re.match(r"\s*def accumulate\(", line):
            in_acc = False

    patched_txt = "".join(out)
    if "import os" not in patched_txt[:500]:
        patched_txt = "import os\n" + patched_txt
    open(coco_fp, "w", encoding="utf-8").write(patched_txt)
    print("✅ Patch applied: predictions will be saved when DFINE_PRED_PATH is set.")


✅ CocoEvaluator file: /content/D-FINE/src/data/dataset/coco_eval.py
✅ Patch applied: predictions will be saved when DFINE_PRED_PATH is set.


In [ ]:
import os
import yaml

# Configuration variables
base_data_path = "/content/drive/MyDrive/PCB_MC/Data/components_only/kfold_data" # Update this
folds = ["fold_0", "fold_1", "fold_2", "fold_3", "fold_4"]
model_config_relative = "configs/dfine/dfine_hgnetv2_l_coco.yml" # Corrected path to config relative to D-FINE root

# Define the D-FINE repository path
dfine_path = "/content/D-FINE"

# Check if D-FINE directory exists from the previous step
if not os.path.exists(dfine_path):
    print(f"Error: D-FINE directory not found at {dfine_path}. Please ensure the cloning step was successful.")
else:
    for fold in folds:
        print(f"\n\ud83d\ude80 Starting Training for {fold}...")



        train_img_folder = os.path.join(base_data_path, fold, 'train', 'images')
        train_ann_file = os.path.join(base_data_path, fold, 'train', 'COCO_train.json')
        val_img_folder = os.path.join(base_data_path, fold, 'valid', 'images')
        val_ann_file = os.path.join(base_data_path, fold, 'valid', 'COCO_valid.json')


        # Trigger D-FINE training by first changing directory to D-FINE and then executing train.py
        # This ensures that train.py resolves its internal paths (like config and output_dir) correctly.
        # We use the --update flag to dynamically set the dataset paths in the configuration.
        command = f"""
        cd {dfine_path} && \
        python train.py \
            -c {model_config_relative} \
            --use-amp \
            --seed 42 \
            -u train_dataloader.dataset.img_folder="{train_img_folder}" \
               train_dataloader.dataset.ann_file="{train_ann_file}" \
               val_dataloader.dataset.img_folder="{val_img_folder}" \
               val_dataloader.dataset.ann_file="{val_ann_file}" \
               remap_mscoco_category=False \
               num_classes=23 \
               train_dataloader.total_batch_size=16 \
               val_dataloader.total_batch_size=32 \
            --output-dir ./output/{fold}
        """
        # Execute the command
        !{command}

        print(f"\u2705 Finished training for {fold}. Weights saved to {dfine_path}/output/{fold}")

ERROR:tornado.application:Exception in callback functools.partial(<bound method OutStream._flush of <ipykernel.iostream.OutStream object at 0x78a01939cd90>>)
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py", line 104, in json_packer
    ).encode("utf8", errors="surrogateescape")
      ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
UnicodeEncodeError: 'utf-8' codec can't encode characters in position 30-31: surrogates not allowed

During handling of the above exception, another exception occurred:

Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/tornado/ioloop.py", line 758, in _run_callback
    ret = callback()
          ^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/ipykernel/iostream.py", line 518, in _flush
    self.session.send(
  File "/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py", line 848, in send
    to_send = self.serialize(msg, ident)
              ^^^^^^

2026-02-14 17:09:53.943238: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-02-14 17:09:53.964023: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1771088993.987817    1596 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1771088993.995125    1596 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1771088994.014551    1596 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking 

ERROR:tornado.application:Exception in callback functools.partial(<bound method OutStream._flush of <ipykernel.iostream.OutStream object at 0x78a01939cd90>>)
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py", line 104, in json_packer
    ).encode("utf8", errors="surrogateescape")
      ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
UnicodeEncodeError: 'utf-8' codec can't encode characters in position 110-111: surrogates not allowed

During handling of the above exception, another exception occurred:

Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/tornado/ioloop.py", line 758, in _run_callback
    ret = callback()
          ^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/ipykernel/iostream.py", line 518, in _flush
    self.session.send(
  File "/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py", line 848, in send
    to_send = self.serialize(msg, ident)
              ^^^^

2026-02-14 17:10:19.690763: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-02-14 17:10:19.709099: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1771089019.731322    1805 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1771089019.738642    1805 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1771089019.757428    1805 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking 

ERROR:tornado.application:Exception in callback functools.partial(<bound method OutStream._flush of <ipykernel.iostream.OutStream object at 0x78a01939cd90>>)
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py", line 104, in json_packer
    ).encode("utf8", errors="surrogateescape")
      ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
UnicodeEncodeError: 'utf-8' codec can't encode characters in position 110-111: surrogates not allowed

During handling of the above exception, another exception occurred:

Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/tornado/ioloop.py", line 758, in _run_callback
    ret = callback()
          ^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/ipykernel/iostream.py", line 518, in _flush
    self.session.send(
  File "/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py", line 848, in send
    to_send = self.serialize(msg, ident)
              ^^^^

2026-02-14 17:10:38.022187: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-02-14 17:10:38.039958: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1771089038.061732    1950 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1771089038.068882    1950 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1771089038.086992    1950 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking 

ERROR:tornado.application:Exception in callback functools.partial(<bound method OutStream._flush of <ipykernel.iostream.OutStream object at 0x78a01939cd90>>)
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py", line 104, in json_packer
    ).encode("utf8", errors="surrogateescape")
      ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
UnicodeEncodeError: 'utf-8' codec can't encode characters in position 110-111: surrogates not allowed

During handling of the above exception, another exception occurred:

Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/tornado/ioloop.py", line 758, in _run_callback
    ret = callback()
          ^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/ipykernel/iostream.py", line 518, in _flush
    self.session.send(
  File "/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py", line 848, in send
    to_send = self.serialize(msg, ident)
              ^^^^

2026-02-14 17:10:56.249495: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-02-14 17:10:56.267454: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1771089056.289487    2088 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1771089056.296749    2088 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1771089056.315381    2088 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking 

ERROR:tornado.application:Exception in callback functools.partial(<bound method OutStream._flush of <ipykernel.iostream.OutStream object at 0x78a01939cd90>>)
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py", line 104, in json_packer
    ).encode("utf8", errors="surrogateescape")
      ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
UnicodeEncodeError: 'utf-8' codec can't encode characters in position 110-111: surrogates not allowed

During handling of the above exception, another exception occurred:

Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/tornado/ioloop.py", line 758, in _run_callback
    ret = callback()
          ^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/ipykernel/iostream.py", line 518, in _flush
    self.session.send(
  File "/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py", line 848, in send
    to_send = self.serialize(msg, ident)
              ^^^^

2026-02-14 17:11:14.259373: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-02-14 17:11:14.277736: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1771089074.299901    2228 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1771089074.307191    2228 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1771089074.325784    2228 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking 

In [3]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


###Subset: Components Only

Classes: 23

# --- Run evaluation + export predictions.json (requires patched CocoEvaluator) ---
ckpt = os.path.join(fold_output_dir, "last.pth")
if not os.path.exists(ckpt):
    ckpt = os.path.join(fold_output_dir, "latest.pth")

pred_path = os.path.join(fold_output_dir, "predictions.json")

test_command = f"""
cd {dfine_path} && \
DFINE_PRED_PATH=\"{pred_path}\" \
python train.py \
    -c {model_config_relative} \
    --test-only \
    -r {ckpt} \
    -u val_dataloader.dataset.img_folder=\"{val_img_folder}\" \
       val_dataloader.dataset.ann_file=\"{val_ann_file}\" \
       remap_mscoco_category=False \
       num_classes=8 \
    --output-dir {fold_output_dir}
"""

!{test_command}
print(f"✅ Finished eval for {fold}. Predictions saved to {pred_path}")


In [ ]:
import os


# Configuration variables
base_data_path = "/content/drive/MyDrive/PCB_MC/Data/components_only/kfold_data"
# Define where you want the weights and logs to be saved in your Drive
base_output_path = "/content/drive/MyDrive/PCB_MC/Results1/D-Fine/components_only"
folds = ["fold_0", "fold_1", "fold_2", "fold_3", "fold_4"]
model_config_relative = "configs/dfine/dfine_hgnetv2_l_coco.yml"
dfine_path = "/content/D-FINE"

# Ensure the output directory exists
if not os.path.exists(base_output_path):
    os.makedirs(base_output_path)
    print(f"Created output directory at {base_output_path}")

if not os.path.exists(dfine_path):
    print(f"Error: D-FINE directory not found at {dfine_path}.")
else:
    for fold in folds:
        print(f"\n🚀 Starting Training for {fold}...")

        # Setup paths for data
        train_img_folder = os.path.join(base_data_path, fold, 'train', 'images')
        train_ann_file = os.path.join(base_data_path, fold, 'train', 'COCO_train.json')
        val_img_folder = os.path.join(base_data_path, fold, 'valid', 'images')
        val_ann_file = os.path.join(base_data_path, fold, 'valid', 'COCO_valid.json')

        # Define the specific output folder for this fold inside Drive
        fold_output_dir = os.path.join(base_output_path, fold)

        command = f"""
        cd {dfine_path} && \
        python train.py \
            -c {model_config_relative} \
            --use-amp \
            --seed 42 \
            -u train_dataloader.dataset.img_folder="{train_img_folder}" \
               train_dataloader.dataset.ann_file="{train_ann_file}" \
               val_dataloader.dataset.img_folder="{val_img_folder}" \
               val_dataloader.dataset.ann_file="{val_ann_file}" \
               remap_mscoco_category=False \
               num_classes=23 \
               train_dataloader.total_batch_size=16 \
               val_dataloader.total_batch_size=32 \
            --output-dir {fold_output_dir}
        """

        # Execute the command
        !{command}

        print(f"✅ Finished training for {fold}. Weights saved to {fold_output_dir}")

Streaming output truncated to the last 5000 lines.
 'ema_decay': 0.9999,
 'ema_warmups': 2000,
 'epochs': 80,
 'find_unused_parameters': False,
 'last_epoch': -1,
 'num_workers': 0,
 'output_dir': '/content/drive/MyDrive/PCB_MC/Results1/D-Fine/components_only/fold_3',
 'print_freq': 100,
 'resume': None,
 'seed': 42,
 'summary_dir': None,
 'sync_bn': True,
 'task': 'detection',
 'tuning': None,
 'use_amp': True,
 'use_ema': True,
 'yaml_cfg': {'DFINE': {'backbone': 'HGNetv2',
                        'decoder': 'DFINETransformer',
                        'encoder': 'HybridEncoder'},
              'DFINECriterion': {'alpha': 0.75,
                                 'gamma': 2.0,
                                 'losses': ['vfl', 'boxes', 'local'],
                                 'matcher': {'alpha': 0.25,
                                             'gamma': 2.0,
                                             'type': 'HungarianMatcher',
                                             'weight_d

###Subset: Full Dataset

Classes: 31

# --- Run evaluation + export predictions.json (requires patched CocoEvaluator) ---
ckpt = os.path.join(fold_output_dir, "last.pth")
if not os.path.exists(ckpt):
    ckpt = os.path.join(fold_output_dir, "latest.pth")

pred_path = os.path.join(fold_output_dir, "predictions.json")

test_command = f"""
cd {dfine_path} && \
DFINE_PRED_PATH=\"{pred_path}\" \
python train.py \
    -c {model_config_relative} \
    --test-only \
    -r {ckpt} \
    -u val_dataloader.dataset.img_folder=\"{val_img_folder}\" \
       val_dataloader.dataset.ann_file=\"{val_ann_file}\" \
       remap_mscoco_category=False \
       num_classes=8 \
    --output-dir {fold_output_dir}
"""

!{test_command}
print(f"✅ Finished eval for {fold}. Predictions saved to {pred_path}")


In [ ]:
import os


# Configuration variables
base_data_path = "/content/drive/MyDrive/PCB_MC/Data/full_dataset/kfold_data"
# Define where you want the weights and logs to be saved in your Drive
base_output_path = "/content/drive/MyDrive/PCB_MC/Results1/D-Fine/full_dataset"
folds = ["fold_0", "fold_1", "fold_2", "fold_3", "fold_4"]
model_config_relative = "configs/dfine/dfine_hgnetv2_l_coco.yml"
dfine_path = "/content/D-FINE"

# Ensure the output directory exists
if not os.path.exists(base_output_path):
    os.makedirs(base_output_path)
    print(f"Created output directory at {base_output_path}")

if not os.path.exists(dfine_path):
    print(f"Error: D-FINE directory not found at {dfine_path}.")
else:
    for fold in folds:
        print(f"\n🚀 Starting Training for {fold}...")

        # Setup paths for data
        train_img_folder = os.path.join(base_data_path, fold, 'train', 'images')
        train_ann_file = os.path.join(base_data_path, fold, 'train', 'COCO_train.json')
        val_img_folder = os.path.join(base_data_path, fold, 'valid', 'images')
        val_ann_file = os.path.join(base_data_path, fold, 'valid', 'COCO_valid.json')

        # Define the specific output folder for this fold inside Drive
        fold_output_dir = os.path.join(base_output_path, fold)

        command = f"""
        cd {dfine_path} && \
        python train.py \
            -c {model_config_relative} \
            --use-amp \
            --seed 42 \
            -u train_dataloader.dataset.img_folder="{train_img_folder}" \
               train_dataloader.dataset.ann_file="{train_ann_file}" \
               val_dataloader.dataset.img_folder="{val_img_folder}" \
               val_dataloader.dataset.ann_file="{val_ann_file}" \
               remap_mscoco_category=False \
               num_classes=31 \
               train_dataloader.total_batch_size=16 \
               val_dataloader.total_batch_size=32 \
            --output-dir {fold_output_dir}
        """

        # Execute the command
        !{command}

        print(f"✅ Finished training for {fold}. Weights saved to {fold_output_dir}")

###Subset: Missing Only

Classes: 8

# --- Run evaluation + export predictions.json (requires patched CocoEvaluator) ---
ckpt = os.path.join(fold_output_dir, "last.pth")
if not os.path.exists(ckpt):
    ckpt = os.path.join(fold_output_dir, "latest.pth")

pred_path = os.path.join(fold_output_dir, "predictions.json")

test_command = f"""
cd {dfine_path} && \
DFINE_PRED_PATH=\"{pred_path}\" \
python train.py \
    -c {model_config_relative} \
    --test-only \
    -r {ckpt} \
    -u val_dataloader.dataset.img_folder=\"{val_img_folder}\" \
       val_dataloader.dataset.ann_file=\"{val_ann_file}\" \
       remap_mscoco_category=False \
       num_classes=8 \
    --output-dir {fold_output_dir}
"""

!{test_command}
print(f"✅ Finished eval for {fold}. Predictions saved to {pred_path}")


In [ ]:
import os


# Configuration variables
base_data_path = "/content/drive/MyDrive/PCB_MC/Data/missing_only/kfold_data"
# Define where you want the weights and logs to be saved in your Drive
base_output_path = "/content/drive/MyDrive/PCB_MC/Results1/D-Fine/missing_only"
folds = ["fold_0", "fold_1", "fold_2", "fold_3", "fold_4"]
model_config_relative = "configs/dfine/dfine_hgnetv2_l_coco.yml"
dfine_path = "/content/D-FINE"

# Ensure the output directory exists
if not os.path.exists(base_output_path):
    os.makedirs(base_output_path)
    print(f"Created output directory at {base_output_path}")

if not os.path.exists(dfine_path):
    print(f"Error: D-FINE directory not found at {dfine_path}.")
else:
    for fold in folds:
        print(f"\n🚀 Starting Training for {fold}...")

        # Setup paths for data
        train_img_folder = os.path.join(base_data_path, fold, 'train', 'images')
        train_ann_file = os.path.join(base_data_path, fold, 'train', 'COCO_train.json')
        val_img_folder = os.path.join(base_data_path, fold, 'valid', 'images')
        val_ann_file = os.path.join(base_data_path, fold, 'valid', 'COCO_valid.json')

        # Define the specific output folder for this fold inside Drive
        fold_output_dir = os.path.join(base_output_path, fold)

        command = f"""
        cd {dfine_path} && \
        python train.py \
            -c {model_config_relative} \
            --use-amp \
            --seed 42 \
            -u train_dataloader.dataset.img_folder="{train_img_folder}" \
               train_dataloader.dataset.ann_file="{train_ann_file}" \
               val_dataloader.dataset.img_folder="{val_img_folder}" \
               val_dataloader.dataset.ann_file="{val_ann_file}" \
               remap_mscoco_category=False \
               num_classes=8 \
               train_dataloader.total_batch_size=16 \
               val_dataloader.total_batch_size=32 \
            --output-dir {fold_output_dir}
        """

        # Execute the command
        !{command}

        print(f"✅ Finished training for {fold}. Weights saved to {fold_output_dir}")

Streaming output truncated to the last 5000 lines.
 'summary_dir': None,
 'sync_bn': True,
 'task': 'detection',
 'tuning': None,
 'use_amp': True,
 'use_ema': True,
 'yaml_cfg': {'DFINE': {'backbone': 'HGNetv2',
                        'decoder': 'DFINETransformer',
                        'encoder': 'HybridEncoder'},
              'DFINECriterion': {'alpha': 0.75,
                                 'gamma': 2.0,
                                 'losses': ['vfl', 'boxes', 'local'],
                                 'matcher': {'alpha': 0.25,
                                             'gamma': 2.0,
                                             'type': 'HungarianMatcher',
                                             'weight_dict': {'cost_bbox': 5,
                                                             'cost_class': 2,
                                                             'cost_giou': 2}},
                                 'reg_max': 32,
                                 'weight

###Subset: Non Missing

Classes: 23

# --- Run evaluation + export predictions.json (requires patched CocoEvaluator) ---
ckpt = os.path.join(fold_output_dir, "last.pth")
if not os.path.exists(ckpt):
    ckpt = os.path.join(fold_output_dir, "latest.pth")

pred_path = os.path.join(fold_output_dir, "predictions.json")

test_command = f"""
cd {dfine_path} && \
DFINE_PRED_PATH=\"{pred_path}\" \
python train.py \
    -c {model_config_relative} \
    --test-only \
    -r {ckpt} \
    -u val_dataloader.dataset.img_folder=\"{val_img_folder}\" \
       val_dataloader.dataset.ann_file=\"{val_ann_file}\" \
       remap_mscoco_category=False \
       num_classes=8 \
    --output-dir {fold_output_dir}
"""

!{test_command}
print(f"✅ Finished eval for {fold}. Predictions saved to {pred_path}")


In [ ]:
import os


# Configuration variables
base_data_path = "/content/drive/MyDrive/PCB_MC/Data/non_missing/kfold_data"
# Define where you want the weights and logs to be saved in your Drive
base_output_path = "/content/drive/MyDrive/PCB_MC/Results1/D-Fine/non_missing"
folds = ["fold_0", "fold_1", "fold_2", "fold_3", "fold_4"]
model_config_relative = "configs/dfine/dfine_hgnetv2_l_coco.yml"
dfine_path = "/content/D-FINE"

# Ensure the output directory exists
if not os.path.exists(base_output_path):
    os.makedirs(base_output_path)
    print(f"Created output directory at {base_output_path}")

if not os.path.exists(dfine_path):
    print(f"Error: D-FINE directory not found at {dfine_path}.")
else:
    for fold in folds:
        print(f"\n🚀 Starting Training for {fold}...")

        # Setup paths for data
        train_img_folder = os.path.join(base_data_path, fold, 'train', 'images')
        train_ann_file = os.path.join(base_data_path, fold, 'train', 'COCO_train.json')
        val_img_folder = os.path.join(base_data_path, fold, 'valid', 'images')
        val_ann_file = os.path.join(base_data_path, fold, 'valid', 'COCO_valid.json')

        # Define the specific output folder for this fold inside Drive
        fold_output_dir = os.path.join(base_output_path, fold)

        command = f"""
        cd {dfine_path} && \
        python train.py \
            -c {model_config_relative} \
            --use-amp \
            --seed 42 \
            -u train_dataloader.dataset.img_folder="{train_img_folder}" \
               train_dataloader.dataset.ann_file="{train_ann_file}" \
               val_dataloader.dataset.img_folder="{val_img_folder}" \
               val_dataloader.dataset.ann_file="{val_ann_file}" \
               remap_mscoco_category=False \
               num_classes=23 \
               train_dataloader.total_batch_size=16 \
               val_dataloader.total_batch_size=32 \
            --output-dir {fold_output_dir}
        """

        # Execute the command
        !{command}

        print(f"✅ Finished training for {fold}. Weights saved to {fold_output_dir}")

Streaming output truncated to the last 5000 lines.
 '_evaluator': None,
 '_lr_scheduler': None,
 '_lr_warmup_scheduler': None,
 '_model': None,
 '_optimizer': None,
 '_postprocessor': None,
 '_scaler': None,
 '_train_batch_size': None,
 '_train_dataloader': None,
 '_train_dataset': None,
 '_train_shuffle': None,
 '_val_batch_size': None,
 '_val_dataloader': None,
 '_val_dataset': None,
 '_val_shuffle': None,
 '_writer': None,
 'batch_size': None,
 'checkpoint_freq': 12,
 'clip_max_norm': 0.1,
 'device': '',
 'ema_decay': 0.9999,
 'ema_warmups': 2000,
 'epochs': 80,
 'find_unused_parameters': False,
 'last_epoch': -1,
 'num_workers': 0,
 'output_dir': '/content/drive/MyDrive/PCB_MC/Results1/D-Fine/non_missing/fold_3',
 'print_freq': 100,
 'resume': None,
 'seed': 42,
 'summary_dir': None,
 'sync_bn': True,
 'task': 'detection',
 'tuning': None,
 'use_amp': True,
 'use_ema': True,
 'yaml_cfg': {'DFINE': {'backbone': 'HGNetv2',
                        'decoder': 'DFINETransformer',
      

### Visualization (optional)
Draw high-contrast boxes from the exported `predictions.json`.

In [ ]:

# --- Quick visualization from COCO predictions.json (high-contrast for green PCBs) ---
import os, json, cv2
from collections import defaultdict

fold_output_dir = "/content/drive/MyDrive/PCB_MC/Results1/D-Fine/missing_only/fold_0"
pred_json = os.path.join(fold_output_dir, "predictions.json")
coco_gt  = "/content/drive/MyDrive/PCB_MC/Data/missing_only/kfold_data/fold_0/valid/COCO_valid.json"
img_dir  = "/content/drive/MyDrive/PCB_MC/Data/missing_only/kfold_data/fold_0/valid/images"
out_dir  = os.path.join(fold_output_dir, "viz")
os.makedirs(out_dir, exist_ok=True)

CONF_TH = 0.80
MAGENTA = (255, 0, 255)  # BGR
ORANGE  = (0, 165, 255)
WHITE   = (255, 255, 255)
BLACK   = (0, 0, 0)

def draw_label(img, text, x, y):
    cv2.putText(img, text, (x, y), cv2.FONT_HERSHEY_SIMPLEX, 0.5, BLACK, 3, cv2.LINE_AA)
    cv2.putText(img, text, (x, y), cv2.FONT_HERSHEY_SIMPLEX, 0.5, WHITE, 1, cv2.LINE_AA)

assert os.path.isfile(pred_json), f"predictions.json not found: {pred_json}"
with open(pred_json, "r") as f:
    preds = json.load(f)

coco = json.load(open(coco_gt, "r"))
id2file = {im["id"]: im["file_name"] for im in coco["images"]}

by_image = defaultdict(list)
for p in preds:
    by_image[p["image_id"]].append(p)

count = 0
for image_id, dets in by_image.items():
    fn = id2file.get(image_id)
    if not fn:
        continue
    img_path = os.path.join(img_dir, fn)
    img = cv2.imread(img_path)
    if img is None:
        continue

    for d in dets:
        x, y, w, h = d["bbox"]
        x1, y1, x2, y2 = int(x), int(y), int(x+w), int(y+h)
        score = float(d["score"])
        cls = int(d["category_id"])

        color = ORANGE if score < CONF_TH else MAGENTA
        cv2.rectangle(img, (x1, y1), (x2, y2), color, 2)

        if score < CONF_TH:
            draw_label(img, f"{cls}:{score:.2f}", x1, max(15, y1-5))

    cv2.imwrite(os.path.join(out_dir, fn), img)
    count += 1
    if count >= 50:
        break

print(f"✅ Wrote {count} visualizations to: {out_dir}")


AssertionError: predictions.json not found: /content/drive/MyDrive/PCB_MC/Results1/D-Fine/missing_only/fold_0/predictions.json

In [4]:
!find "/content/drive/MyDrive/PCB_MC/Results1/D-Fine" -maxdepth 6 -type f -name "*pred*.json" -o -name "*result*.json" -o -name "*bbox*.json"


In [6]:
import os, re, glob, textwrap

dfine_path = "/content/D-FINE"

# Search for coco evaluator python file(s)
candidates = []
for pat in ["**/*coco*evaluator*.py", "**/*coco_eval*.py", "**/*evaluator*.py"]:
    candidates += glob.glob(os.path.join(dfine_path, pat), recursive=True)

# Prefer files that mention COCOeval / CocoEvaluator
ranked = []
for fp in set(candidates):
    try:
        s = open(fp, "r", encoding="utf-8").read()
        if ("COCOeval" in s) or ("CocoEvaluator" in s) or ("coco_eval" in s):
            ranked.append(fp)
    except:
        pass

print("Found evaluator candidates:")
for fp in ranked[:20]:
    print(" -", fp)

assert ranked, "Couldn't find COCO evaluator file. Paste your /content/D-FINE tree."

# Pick first best candidate
evaluator_fp = ranked[0]
print("\nUsing:", evaluator_fp)

code = open(evaluator_fp, "r", encoding="utf-8").read()

# If already patched, don't double patch
if "DFINE_DUMP_PREDICTIONS_JSON" in code:
    print("✅ Evaluator already patched.")
else:
    # We inject a helper and a dump call where predictions exist.
    # Many evaluators store predictions in self.predictions, self.img_ids, or results dict.
    # We'll add a safe method: if the evaluator has self.predictions (list), dump it.
    inject_helper = textwrap.dedent("""
    # --- DFINE_DUMP_PREDICTIONS_JSON (added for PCB visualization) ---
    import os as _os
    import json as _json

    def _dfine_dump_predictions_json(obj, output_dir):
        \"\"\"Try to dump COCO-format detections if present on evaluator.\"\"\"
        out_path = _os.path.join(output_dir, "predictions.json")

        # Common storage patterns in COCO evaluators
        for attr in ["predictions", "preds", "_predictions", "coco_results", "results"]:
            if hasattr(obj, attr):
                data = getattr(obj, attr)
                # Some evaluators store dict per iou_type; try bbox
                if isinstance(data, dict) and "bbox" in data:
                    data = data["bbox"]
                if isinstance(data, (list, tuple)) and len(data) > 0 and isinstance(data[0], dict) and "bbox" in data[0]:
                    with open(out_path, "w") as f:
                        _json.dump(list(data), f)
                    print(f"✅ Saved predictions.json to {out_path} ({len(data)} dets)")
                    return True

        # Sometimes stored as per-image mapping: {image_id: [dets]}
        for attr in ["predictions_by_image", "by_image", "_by_image"]:
            if hasattr(obj, attr):
                mp = getattr(obj, attr)
                if isinstance(mp, dict):
                    flat = []
                    for _, dets in mp.items():
                        if isinstance(dets, list):
                            flat.extend(dets)
                    if len(flat) > 0 and isinstance(flat[0], dict) and "bbox" in flat[0]:
                        with open(out_path, "w") as f:
                            _json.dump(flat, f)
                        print(f"✅ Saved predictions.json to {out_path} ({len(flat)} dets)")
                        return True

        return False
    # --- end DFINE_DUMP_PREDICTIONS_JSON ---
    """)

    # Put helper near top (after imports). We insert after first import block.
    m = re.search(r"(\\n\\s*import[^\\n]+\\n)", code)
    insert_at = m.end() if m else 0
    code2 = code[:insert_at] + "\n" + inject_helper + "\n" + code[insert_at:]

    # Now add a dump call at the end of evaluation (look for summarize/accumulate or end of evaluate)
    # We'll try to hook after "accumulate()" or after "summarize()".
    hook_patterns = ["summarize()", "accumulate()"]
    hooked = False
    for pat in hook_patterns:
        idx = code2.rfind(pat)
        if idx != -1:
            # insert after that line
            line_end = code2.find("\n", idx)
            if line_end != -1:
                inject_call = "\n    _dfine_dump_predictions_json(self, getattr(self, 'output_dir', output_dir) if 'output_dir' in locals() else getattr(self, 'output_dir', '.'))\n"
                code2 = code2[:line_end+1] + inject_call + code2[line_end+1:]
                hooked = True
                break

    if not hooked:
        # Fallback: append note; you can hook manually later
        print("⚠️ Could not auto-hook dump call; evaluator structure is unusual.")
        print("Open the evaluator file and add _dfine_dump_predictions_json(self, output_dir) at the end of evaluate().")

    with open(evaluator_fp, "w", encoding="utf-8") as f:
        f.write(code2)

    print("✅ Patched evaluator to dump predictions.json (best-effort hook).")


Found evaluator candidates:
 - /content/D-FINE/src/data/dataset/coco_eval.py

Using: /content/D-FINE/src/data/dataset/coco_eval.py
⚠️ Could not auto-hook dump call; evaluator structure is unusual.
Open the evaluator file and add _dfine_dump_predictions_json(self, output_dir) at the end of evaluate().
✅ Patched evaluator to dump predictions.json (best-effort hook).


In [11]:
dfine_path="/content/D-FINE"
fold_output_dir="/content/drive/MyDrive/PCB_MC/Results/D-Fine/missing_only/fold_0"
val_img_folder="/content/drive/MyDrive/PCB_MC/Data/missing_only/kfold_data/fold_0/valid/images"
val_ann_file="/content/drive/MyDrive/PCB_MC/Data/missing_only/kfold_data/fold_0/valid/COCO_valid.json"
model_config_relative="configs/dfine/dfine_hgnetv2_l_coco.yml"

pred_path = os.path.join(fold_output_dir, "predictions.json")

command = f"""
cd {dfine_path} && \
DFINE_PRED_PATH=\"{pred_path}\" \
python train.py \
  -c {model_config_relative} \
  --test-only \
  -r {fold_output_dir}/last.pth \
  -u val_dataloader.dataset.img_folder=\"{val_img_folder}\" \
     val_dataloader.dataset.ann_file=\"{val_ann_file}\" \
     remap_mscoco_category=False \
     num_classes=8 \
  --output-dir {fold_output_dir}
"""
!{command}

print(f"✅ Finished eval for {fold_output_dir}. Predictions saved to {pred_path}")

2026-02-16 10:06:33.100378: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-02-16 10:06:33.119981: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1771236393.142455    4616 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1771236393.150333    4616 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1771236393.169795    4616 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking 

In [8]:
!ls -lah "/content/drive/MyDrive/PCB_MC/Results/D-Fine/missing_only/fold_0" | sed -n '1,200p'
!ls -lah "/content/drive/MyDrive/PCB_MC/Results/D-Fine/missing_only/fold_0/eval" | sed -n '1,200p'


total 3.8G
-rw------- 1 root root 477M Feb  1 17:13 best_stg1.pth
-rw------- 1 root root 477M Feb  1 16:29 checkpoint0011.pth
-rw------- 1 root root 477M Feb  1 16:38 checkpoint0023.pth
-rw------- 1 root root 477M Feb  1 16:47 checkpoint0035.pth
-rw------- 1 root root 477M Feb  1 16:56 checkpoint0047.pth
-rw------- 1 root root 477M Feb  1 17:05 checkpoint0059.pth
-rw------- 1 root root 477M Feb  1 17:14 checkpoint0071.pth
drwx------ 2 root root 4.0K Feb  1 16:19 eval
-rw------- 1 root root 1.7M Feb 16 10:00 eval.pth
-rw------- 1 root root 477M Feb  1 17:14 last.pth
-rw------- 1 root root 263K Feb  1 17:16 log.txt
drwx------ 2 root root 4.0K Feb 16 10:00 summary
drwx------ 2 root root 4.0K Feb  1 16:18 train_samples
drwx------ 2 root root 4.0K Feb  1 16:19 val_samples
total 5.0M
-rw------- 1 root root 1.6M Feb  1 16:19 000.pth
-rw------- 1 root root 1.7M Feb  1 16:58 050.pth
-rw------- 1 root root 1.8M Feb  1 17:16 latest.pth


In [12]:
import os, glob, torch

FOLD_DIR = "/content/drive/MyDrive/PCB_MC/Results/D-Fine/missing_only/fold_0"
eval_files = sorted(glob.glob(os.path.join(FOLD_DIR, "eval", "*.pth")))
print("Eval files:")
for fp in eval_files:
    print(" -", os.path.basename(fp), f"({os.path.getsize(fp)/1024:.1f} KB)")

def preview(fp):
    obj = torch.load(fp, map_location="cpu", weights_only=False)
    print("\n===", os.path.basename(fp), "===")
    print("type:", type(obj))
    if isinstance(obj, dict):
        print("keys:", list(obj.keys())[:30])
        # print a small preview of one value
        k = list(obj.keys())[0]
        v = obj[k]
        print("sample key:", k, "| sample value type:", type(v))
    elif isinstance(obj, list):
        print("len:", len(obj))
        print("first type:", type(obj[0]) if len(obj) else None)
        if len(obj) and isinstance(obj[0], dict):
            print("first keys:", list(obj[0].keys())[:30])

for fp in eval_files[:3]:
    preview(fp)


Eval files:
 - 000.pth (1622.6 KB)
 - 050.pth (1716.2 KB)
 - latest.pth (1747.4 KB)

=== 000.pth ===
type: <class 'dict'>
keys: ['params', 'counts', 'date', 'matched', 'precision', 'scores', 'recall', 'evaluations_size']
sample key: params | sample value type: <class 'faster_coco_eval.core.cocoeval.Params'>

=== 050.pth ===
type: <class 'dict'>
keys: ['params', 'counts', 'date', 'matched', 'precision', 'scores', 'recall', 'evaluations_size']
sample key: params | sample value type: <class 'faster_coco_eval.core.cocoeval.Params'>

=== latest.pth ===
type: <class 'dict'>
keys: ['params', 'counts', 'date', 'matched', 'precision', 'scores', 'recall', 'evaluations_size']
sample key: params | sample value type: <class 'faster_coco_eval.core.cocoeval.Params'>


In [15]:
import os, re

dfine_path = "/content/D-FINE"
solver_fp = os.path.join(dfine_path, "src", "solver", "_solver.py")
assert os.path.exists(solver_fp), f"Not found: {solver_fp}"

code = open(solver_fp, "r", encoding="utf-8").read()

# If already patched, skip
if "DFINE_SAVE_PREDICTIONS_JSON" in code:
    print("✅ _solver.py already patched for predictions export.")
else:
    # 1) Inject helper function near top (after imports)
    helper = r'''
# --- DFINE_SAVE_PREDICTIONS_JSON (added for PCB visualization) ---
import json as _json
import os as _os

def _dfine_append_coco_preds(buffer, targets, results):
    """
    targets: list[dict] each with 'image_id'
    results: list[dict] each with 'boxes'(xyxy), 'scores', 'labels'
    """
    for tgt, res in zip(targets, results):
        image_id = int(tgt["image_id"])
        boxes = res["boxes"]
        scores = res["scores"]
        labels = res["labels"]

        # tensor -> list
        if hasattr(boxes, "tolist"): boxes = boxes.tolist()
        if hasattr(scores, "tolist"): scores = scores.tolist()
        if hasattr(labels, "tolist"): labels = labels.tolist()

        for (x1, y1, x2, y2), s, lab in zip(boxes, scores, labels):
            buffer.append({
                "image_id": image_id,
                "category_id": int(lab),
                "bbox": [float(x1), float(y1), float(x2 - x1), float(y2 - y1)],  # xywh
                "score": float(s),
            })

def _dfine_dump_coco_preds(buffer, output_dir):
    out_path = _os.path.join(output_dir, "predictions.json")
    with open(out_path, "w") as f:
        _json.dump(buffer, f)
    print(f"✅ Saved predictions.json: {out_path} ({len(buffer)} detections)")
# --- end DFINE_SAVE_PREDICTIONS_JSON ---
'''

    # Insert helper after the last import block near the top
    # We insert after the first blank line following imports.
    m = re.search(r"(\n\s*\n)", code[:4000])  # early in file
    insert_at = m.end() if m else 0
    code = code[:insert_at] + helper + "\n" + code[insert_at:]

    # 2) Hook into test/eval loop:
    # Find a line where postprocessor is called. Common patterns:
    #   results = self.postprocessor(outputs, orig_target_sizes)
    #   results = self._postprocessor(outputs, ...)
    patterns = [
        r"results\s*=\s*self\.postprocessor\([^\n]*\)\s*",
        r"results\s*=\s*self\._postprocessor\([^\n]*\)\s*",
        r"results\s*=\s*self\.post_processor\([^\n]*\)\s*",
    ]

    hooked = False
    for pat in patterns:
        match = re.search(pat, code)
        if match:
            hook_pos = match.end()
            hook_code = "\n        # DFINE_SAVE_PREDICTIONS_JSON\n        if not hasattr(self, '_dfine_pred_buffer'):\n            self._dfine_pred_buffer = []\n        _dfine_append_coco_preds(self._dfine_pred_buffer, targets, results)\n"
            code = code[:hook_pos] + hook_code + code[hook_pos:]
            hooked = True
            print(f"✅ Hooked predictions buffer after postprocessor using pattern: {pat}")
            break

    if not hooked:
        raise RuntimeError(
            "Could not find postprocessor call in _solver.py to hook predictions export.\n"
            "We need to search the exact line in your repo. Run:\n"
            "!grep -n \"postprocessor\" -n /content/D-FINE/src/solver/_solver.py | head -n 50"
        )

    # 3) Dump at the end of test/eval
    # Find end of test-only routine: look for 'Accumulating evaluation results' or 'coco_eval' summarize,
    # and dump after it. More robust: dump right before returning from a test function.
    # We'll insert before the last "return" in the file's test/eval function block by a simple heuristic:
    # insert before the final occurrence of "return" in file.
    ret_idx = code.rfind("\n        return")
    if ret_idx == -1:
        ret_idx = code.rfind("\n    return")
    if ret_idx != -1:
        dump_code = "\n        # DFINE_SAVE_PREDICTIONS_JSON\n        if hasattr(self, '_dfine_pred_buffer'):\n            _dfine_dump_coco_preds(self._dfine_pred_buffer, self.cfg.output_dir)\n"
        code = code[:ret_idx] + dump_code + code[ret_idx:]
        print("✅ Inserted dump call before return.")
    else:
        print("⚠️ Could not find a return statement to hook dump. Predictions may still buffer but not save.")

    open(solver_fp, "w", encoding="utf-8").write(code)
    print("✅ Patched _solver.py successfully.")


RuntimeError: Could not find postprocessor call in _solver.py to hook predictions export.
We need to search the exact line in your repo. Run:
!grep -n "postprocessor" -n /content/D-FINE/src/solver/_solver.py | head -n 50

In [16]:
dfine_path="/content/D-FINE"
fold_output_dir="/content/drive/MyDrive/PCB_MC/Results/D-Fine/missing_only/fold_0"
val_img_folder="/content/drive/MyDrive/PCB_MC/Data/missing_only/kfold_data/fold_0/valid/images"
val_ann_file="/content/drive/MyDrive/PCB_MC/Data/missing_only/kfold_data/fold_0/valid/COCO_valid.json"
model_config_relative="configs/dfine/dfine_hgnetv2_l_coco.yml"

command = f"""
cd {dfine_path} && \
python train.py \
  -c {model_config_relative} \
  --test-only \
  -r {fold_output_dir}/last.pth \
  -u val_dataloader.dataset.img_folder="{val_img_folder}" \
     val_dataloader.dataset.ann_file="{val_ann_file}" \
     remap_mscoco_category=False \
     num_classes=8 \
  --output-dir {fold_output_dir}
"""
!{command}

!ls -lah {fold_output_dir} | head -n 50


2026-02-16 10:11:17.246208: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-02-16 10:11:17.264820: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1771236677.287263    5895 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1771236677.294675    5895 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1771236677.313785    5895 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking 

In [17]:
import os, re

dfine_path = "/content/D-FINE"
solver_fp = os.path.join(dfine_path, "src", "solver", "_solver.py")
assert os.path.exists(solver_fp), f"Not found: {solver_fp}"

code = open(solver_fp, "r", encoding="utf-8").read()

# If already patched, skip
if "DFINE_SAVE_PREDICTIONS_JSON" in code:
    print("✅ _solver.py already patched for predictions export.")
else:
    # 1) Inject helper function near top (after imports)
    helper = r'''
# --- DFINE_SAVE_PREDICTIONS_JSON (added for PCB visualization) ---
import json as _json
import os as _os

def _dfine_append_coco_preds(buffer, targets, results):
    """
    targets: list[dict] each with 'image_id'
    results: list[dict] each with 'boxes'(xyxy), 'scores', 'labels'
    """
    for tgt, res in zip(targets, results):
        image_id = int(tgt["image_id"])
        boxes = res["boxes"]
        scores = res["scores"]
        labels = res["labels"]

        # tensor -> list
        if hasattr(boxes, "tolist"): boxes = boxes.tolist()
        if hasattr(scores, "tolist"): scores = scores.tolist()
        if hasattr(labels, "tolist"): labels = labels.tolist()

        for (x1, y1, x2, y2), s, lab in zip(boxes, scores, labels):
            buffer.append({
                "image_id": image_id,
                "category_id": int(lab),
                "bbox": [float(x1), float(y1), float(x2 - x1), float(y2 - y1)],  # xywh
                "score": float(s),
            })

def _dfine_dump_coco_preds(buffer, output_dir):
    out_path = _os.path.join(output_dir, "predictions.json")
    with open(out_path, "w") as f:
        _json.dump(buffer, f)
    print(f"✅ Saved predictions.json: {out_path} ({len(buffer)} detections)")
# --- end DFINE_SAVE_PREDICTIONS_JSON ---
'''

    # Insert helper after the last import block near the top
    # We insert after the first blank line following imports.
    m = re.search(r"(\n\s*\n)", code[:4000])  # early in file
    insert_at = m.end() if m else 0
    code = code[:insert_at] + helper + "\n" + code[insert_at:]

    # 2) Hook into test/eval loop:
    # Find a line where postprocessor is called. Common patterns:
    #   results = self.postprocessor(outputs, orig_target_sizes)
    #   results = self._postprocessor(outputs, ...)
    patterns = [
        r"results\s*=\s*self\.postprocessor\([^\n]*\)\s*",
        r"results\s*=\s*self\._postprocessor\([^\n]*\)\s*",
        r"results\s*=\s*self\.post_processor\([^\n]*\)\s*",
    ]

    hooked = False
    for pat in patterns:
        match = re.search(pat, code)
        if match:
            hook_pos = match.end()
            hook_code = "\n        # DFINE_SAVE_PREDICTIONS_JSON\n        if not hasattr(self, '_dfine_pred_buffer'):\n            self._dfine_pred_buffer = []\n        _dfine_append_coco_preds(self._dfine_pred_buffer, targets, results)\n"
            code = code[:hook_pos] + hook_code + code[hook_pos:]
            hooked = True
            print(f"✅ Hooked predictions buffer after postprocessor using pattern: {pat}")
            break

    if not hooked:
        raise RuntimeError(
            "Could not find postprocessor call in _solver.py to hook predictions export.\n"
            "We need to search the exact line in your repo. Run:\n"
            "!grep -n \"postprocessor\" -n /content/D-FINE/src/solver/_solver.py | head -n 50"
        )

    # 3) Dump at the end of test/eval
    # Find end of test-only routine: look for 'Accumulating evaluation results' or 'coco_eval' summarize,
    # and dump after it. More robust: dump right before returning from a test function.
    # We'll insert before the last "return" in the file's test/eval function block by a simple heuristic:
    # insert before the final occurrence of "return" in file.
    ret_idx = code.rfind("\n        return")
    if ret_idx == -1:
        ret_idx = code.rfind("\n    return")
    if ret_idx != -1:
        dump_code = "\n        # DFINE_SAVE_PREDICTIONS_JSON\n        if hasattr(self, '_dfine_pred_buffer'):\n            _dfine_dump_coco_preds(self._dfine_pred_buffer, self.cfg.output_dir)\n"
        code = code[:ret_idx] + dump_code + code[ret_idx:]
        print("✅ Inserted dump call before return.")
    else:
        print("⚠️ Could not find a return statement to hook dump. Predictions may still buffer but not save.")

    open(solver_fp, "w", encoding="utf-8").write(code)
    print("✅ Patched _solver.py successfully.")


RuntimeError: Could not find postprocessor call in _solver.py to hook predictions export.
We need to search the exact line in your repo. Run:
!grep -n "postprocessor" -n /content/D-FINE/src/solver/_solver.py | head -n 50

In [18]:
!grep -n "postprocessor" -n /content/D-FINE/src/solver/_solver.py | head -n 50
!grep -n "post" /content/D-FINE/src/solver/_solver.py | head -n 120
!grep -n "bbox" /content/D-FINE/src/solver/_solver.py | head -n 120


137:        self.postprocessor = self.to(cfg.postprocessor, device)
137:        self.postprocessor = self.to(cfg.postprocessor, device)


In [20]:
!grep -RIn "postprocessor(" /content/D-FINE/src | head -n 50
!grep -RIn "self\.postprocessor" /content/D-FINE/src | head -n 80

/content/D-FINE/src/solver/det_engine.py:201:        results = postprocessor(outputs, orig_target_sizes)
/content/D-FINE/src/core/yaml_config.py:43:    def postprocessor(self) -> torch.nn.Module:
/content/D-FINE/src/core/_config.py:87:    def postprocessor(self) -> nn.Module:
/content/D-FINE/src/core/_config.py:91:    def postprocessor(self, m):
/content/D-FINE/src/solver/_solver.py:137:        self.postprocessor = self.to(cfg.postprocessor, device)
/content/D-FINE/src/solver/det_solver.py:48:                self.postprocessor,
/content/D-FINE/src/solver/det_solver.py:111:                self.postprocessor,
/content/D-FINE/src/solver/det_solver.py:215:            self.postprocessor,


In [21]:
import os, re, textwrap

engine_fp = "/content/D-FINE/src/solver/det_engine.py"
assert os.path.exists(engine_fp), f"Not found: {engine_fp}"

code = open(engine_fp, "r", encoding="utf-8").read()

if "DFINE_SAVE_PREDICTIONS_JSON" in code:
    print("✅ det_engine.py already patched.")
else:
    helper = textwrap.dedent("""
    # --- DFINE_SAVE_PREDICTIONS_JSON (added for PCB visualization) ---
    import json as _json
    import os as _os

    def _dfine_append_coco_preds(buffer, targets, results):
        \"\"\"targets: list of dicts w/ image_id; results: list of dicts w/ boxes(xyxy), scores, labels\"\"\"
        for tgt, res in zip(targets, results):
            image_id = int(tgt["image_id"])
            boxes = res["boxes"]
            scores = res["scores"]
            labels = res["labels"]

            if hasattr(boxes, "tolist"): boxes = boxes.tolist()
            if hasattr(scores, "tolist"): scores = scores.tolist()
            if hasattr(labels, "tolist"): labels = labels.tolist()

            for (x1, y1, x2, y2), s, lab in zip(boxes, scores, labels):
                buffer.append({
                    "image_id": image_id,
                    "category_id": int(lab),
                    "bbox": [float(x1), float(y1), float(x2-x1), float(y2-y1)],  # xywh
                    "score": float(s),
                })

    def _dfine_dump_coco_preds(buffer, output_dir):
        out_path = _os.path.join(output_dir, "predictions.json")
        with open(out_path, "w") as f:
            _json.dump(buffer, f)
        print(f"✅ Saved predictions.json: {out_path} ({len(buffer)} detections)")
    # --- end DFINE_SAVE_PREDICTIONS_JSON ---
    """)

    # Insert helper after initial imports block (best effort)
    m = re.search(r"\n\s*\n", code[:4000])
    insert_at = m.end() if m else 0
    code = code[:insert_at] + helper + "\n" + code[insert_at:]

    # Hook after: results = postprocessor(outputs, orig_target_sizes)
    pat = r"results\s*=\s*postprocessor\s*\(\s*outputs\s*,\s*orig_target_sizes\s*\)"
    match = re.search(pat, code)
    if not match:
        raise RuntimeError("Could not find the postprocessor call to hook in det_engine.py")

    line_end = code.find("\n", match.end())
    hook = textwrap.dedent("""
        # DFINE_SAVE_PREDICTIONS_JSON: collect detections
        if not hasattr(postprocessor, "_dfine_pred_buffer"):
            postprocessor._dfine_pred_buffer = []
        _dfine_append_coco_preds(postprocessor._dfine_pred_buffer, targets, results)
    """)
    code = code[:line_end+1] + hook + code[line_end+1:]

    # Dump once at the end of the evaluate/test function:
    # Insert right before the final "return" inside the file (best-effort).
    # This works because det_engine typically has a single test/eval function that returns stats.
    ret_idx = code.rfind("\n    return")
    if ret_idx == -1:
        ret_idx = code.rfind("\n        return")
    if ret_idx == -1:
        print("⚠️ Could not find a return statement for dump hook. We'll append dump at end of file (less ideal).")
        code += textwrap.dedent("""

        # DFINE_SAVE_PREDICTIONS_JSON: dump buffer (fallback)
        try:
            if hasattr(postprocessor, "_dfine_pred_buffer"):
                _dfine_dump_coco_preds(postprocessor._dfine_pred_buffer, output_dir)
        except Exception as _e:
            print("⚠️ Could not dump predictions.json:", _e)
        """)
    else:
        dump = textwrap.dedent("""

            # DFINE_SAVE_PREDICTIONS_JSON: dump buffer once per evaluation
            try:
                if hasattr(postprocessor, "_dfine_pred_buffer"):
                    _dfine_dump_coco_preds(postprocessor._dfine_pred_buffer, output_dir)
            except Exception as _e:
                print("⚠️ Could not dump predictions.json:", _e)
        """)
        code = code[:ret_idx] + dump + code[ret_idx:]

    open(engine_fp, "w", encoding="utf-8").write(code)
    print("✅ Patched det_engine.py to export predictions.json")


✅ Patched det_engine.py to export predictions.json


In [23]:
!cd /content/D-FINE && git checkout -- src/solver/det_engine.py


In [1]:
import os

# Clean up any previous attempts and clone the official D-FINE repository
!rm -rf D-FINE
!git clone https://github.com/Peterande/D-FINE

# Check if the D-FINE directory was created
if os.path.exists('D-FINE'):
    %cd D-FINE
    # Install dependencies
    !pip install -r requirements.txt
    !pip install -U lycoris-lora # Often needed for specific fine-tuning tasks
else:
    print("Error: D-FINE repository could not be cloned. Please check your network connection or the repository URL and try again.")

Cloning into 'D-FINE'...
remote: Enumerating objects: 1401, done.
remote: Counting objects: 100% (658/658), done.
remote: Compressing objects: 100% (206/206), done.
Receiving objects: 100% (1401/1401), 471.66 KiB | 24.82 MiB/s, done.
remote: Total 1401 (delta 514), reused 452 (delta 452), pack-reused 743 (from 3)
Resolving deltas: 100% (890/890), done.
/content/D-FINE


In [2]:
!python -c "import importlib.util; spec=importlib.util.spec_from_file_location('x','/content/D-FINE/src/solver/det_engine.py'); m=importlib.util.module_from_spec(spec); spec.loader.exec_module(m); print('OK')"


2026-02-16 10:22:21.379294: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-02-16 10:22:21.398120: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1771237341.420770    9018 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1771237341.428255    9018 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1771237341.447575    9018 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking 

In [2]:
import os, re, textwrap

engine_fp = "/content/D-FINE/src/solver/det_engine.py"
assert os.path.exists(engine_fp), f"Not found: {engine_fp}"

lines = open(engine_fp, "r", encoding="utf-8").read().splitlines(True)

# If already patched, don't double patch
if any("DFINE_SAVE_PREDICTIONS_JSON" in ln for ln in lines):
    print("✅ det_engine.py already patched (marker found).")
else:
    # 1) Insert helper near the top (after imports block)
    helper = textwrap.dedent("""
    # --- DFINE_SAVE_PREDICTIONS_JSON (added for PCB visualization) ---
    import json as _json
    import os as _os

    def _dfine_append_coco_preds(buffer, targets, results):
        for tgt, res in zip(targets, results):
            image_id = int(tgt["image_id"])
            boxes = res["boxes"]
            scores = res["scores"]
            labels = res["labels"]

            if hasattr(boxes, "tolist"): boxes = boxes.tolist()
            if hasattr(scores, "tolist"): scores = scores.tolist()
            if hasattr(labels, "tolist"): labels = labels.tolist()

            for (x1, y1, x2, y2), s, lab in zip(boxes, scores, labels):
                buffer.append({
                    "image_id": image_id,
                    "category_id": int(lab),
                    "bbox": [float(x1), float(y1), float(x2-x1), float(y2-y1)],  # xywh
                    "score": float(s),
                })

    def _dfine_dump_coco_preds(buffer, output_dir):
        out_path = _os.path.join(output_dir, "predictions.json")
        with open(out_path, "w") as f:
            _json.dump(buffer, f)
        print(f"✅ Saved predictions.json: {out_path} ({len(buffer)} detections)")
    # --- end DFINE_SAVE_PREDICTIONS_JSON ---
    """)

    # Find a good insertion point: after the first blank line after imports
    insert_at = 0
    for i in range(min(len(lines), 3000)):
        if lines[i].strip() == "" and i > 0:
            insert_at = i + 1
            break
    lines.insert(insert_at, helper + "\n")

    # 2) Hook after results = postprocessor(outputs, orig_target_sizes)
    hook_idx = None
    for i, ln in enumerate(lines):
        if "results = postprocessor(outputs, orig_target_sizes)" in ln.replace(" ", ""):
            hook_idx = i
            break
    # The above might miss due to whitespace; use a regex fallback:
    if hook_idx is None:
        rx = re.compile(r"^\s*results\s*=\s*postprocessor\s*\(\s*outputs\s*,\s*orig_target_sizes\s*\)\s*$")
        for i, ln in enumerate(lines):
            if rx.match(ln):
                hook_idx = i
                break

    if hook_idx is None:
        raise RuntimeError("Couldn't find the postprocessor call line in det_engine.py")

    indent = re.match(r"^(\s*)", lines[hook_idx]).group(1)

    hook = (
        f"{indent}# DFINE_SAVE_PREDICTIONS_JSON: collect detections\n"
        f"{indent}if not hasattr(postprocessor, '_dfine_pred_buffer'):\n"
        f"{indent}    postprocessor._dfine_pred_buffer = []\n"
        f"{indent}_dfine_append_coco_preds(postprocessor._dfine_pred_buffer, targets, results)\n"
    )

    lines.insert(hook_idx + 1, hook)

    # 3) Dump before return inside evaluate()
    # Find the LAST 'return' in the file that is indented (likely inside evaluate).
    ret_idx = None
    for i in range(len(lines) - 1, -1, -1):
        if re.match(r"^\s*return\b", lines[i]):
            ret_idx = i
            break

    if ret_idx is None:
        raise RuntimeError("Couldn't find a return statement to hook dump")

    ret_indent = re.match(r"^(\s*)", lines[ret_idx]).group(1)

    dump = (
        f"{ret_indent}# DFINE_SAVE_PREDICTIONS_JSON: dump once per evaluation\n"
        f"{ret_indent}if hasattr(postprocessor, '_dfine_pred_buffer'):\n"
        f"{ret_indent}    _dfine_dump_coco_preds(postprocessor._dfine_pred_buffer, output_dir)\n"
    )

    lines.insert(ret_idx, dump)

    open(engine_fp, "w", encoding="utf-8").write("".join(lines))
    print("✅ Patched det_engine.py (indent-safe).")


✅ Patched det_engine.py (indent-safe).


In [4]:
dfine_path="/content/D-FINE"
fold_output_dir="/content/drive/MyDrive/PCB_MC/Results/D-Fine/missing_only/fold_0"
val_img_folder="/content/drive/MyDrive/PCB_MC/Data/missing_only/kfold_data/fold_0/valid/images"
val_ann_file="/content/drive/MyDrive/PCB_MC/Data/missing_only/kfold_data/fold_0/valid/COCO_valid.json"
model_config_relative="configs/dfine/dfine_hgnetv2_l_coco.yml"

command = f"""
cd {dfine_path} && \
python train.py \
  -c {model_config_relative} \
  --test-only \
  -r {fold_output_dir}/last.pth \
  -u val_dataloader.dataset.img_folder="{val_img_folder}" \
     val_dataloader.dataset.ann_file="{val_ann_file}" \
     remap_mscoco_category=False \
     num_classes=8 \
  --output-dir {fold_output_dir}
"""
!{command}

!ls -lah {fold_output_dir} | head -n 50


2026-02-16 10:27:38.663385: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-02-16 10:27:38.681460: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1771237658.703782    5873 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1771237658.711026    5873 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1771237658.729888    5873 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking 

In [18]:
import re

engine_fp = "/content/D-FINE/src/solver/det_engine.py"
code = open(engine_fp, "r", encoding="utf-8").read()

# Replace the body of _dfine_dump_coco_preds with a robust version
pattern = r"def _dfine_dump_coco_preds\(buffer, output_dir\):[\s\S]*?# --- end DFINE_SAVE_PREDICTIONS_JSON ---"
m = re.search(pattern, code)
assert m, "Couldn't find the DFINE_SAVE_PREDICTIONS_JSON helper block. Did the earlier patch run?"

# Build a safer helper block (keeps the marker)
safe_block = r"""
# --- DFINE_SAVE_PREDICTIONS_JSON (added for PCB visualization) ---
import json as _json
import os as _os

def _dfine_append_coco_preds(buffer, targets, results):
    for tgt, res in zip(targets, results):
        image_id = int(tgt["image_id"])
        boxes = res["boxes"]
        scores = res["scores"]
        labels = res["labels"]

        if hasattr(boxes, "tolist"): boxes = boxes.tolist()
        if hasattr(scores, "tolist"): scores = scores.tolist()
        if hasattr(labels, "tolist"): labels = labels.tolist()

        for (x1, y1, x2, y2), s, lab in zip(boxes, scores, labels):
            buffer.append({
                "image_id": image_id,
                "category_id": int(lab),
                "bbox": [float(x1), float(y1), float(x2-x1), float(y2-y1)],  # xywh
                "score": float(s),
            })

def _dfine_dump_coco_preds(buffer, output_dir):
    # Some forks call evaluate(output_dir=None). Fall back safely:
    if output_dir is None:
        output_dir = _os.environ.get("DFINE_PRED_DIR") or _os.getcwd()

    _os.makedirs(output_dir, exist_ok=True)
    out_path = _os.path.join(output_dir, "predictions.json")
    with open(out_path, "w") as f:
        _json.dump(buffer, f)
    print(f"✅ Saved predictions.json: {out_path} ({len(buffer)} detections)")
# --- end DFINE_SAVE_PREDICTIONS_JSON ---
"""

code2 = code[:m.start()] + safe_block + code[m.end():]
open(engine_fp, "w", encoding="utf-8").write(code2)
print("✅ Updated _dfine_dump_coco_preds to handle output_dir=None using DFINE_PRED_DIR fallback.")


✅ Updated _dfine_dump_coco_preds to handle output_dir=None using DFINE_PRED_DIR fallback.


In [20]:
dfine_path="/content/D-FINE"
fold_output_dir="/content/drive/MyDrive/PCB_MC/Results/D-Fine/non_missing/fold_0"
val_img_folder="/content/drive/MyDrive/PCB_MC/Data/non_missing/kfold_data/fold_0/valid/images"
val_ann_file="/content/drive/MyDrive/PCB_MC/Data/non_missing/kfold_data/fold_0/valid/COCO_valid.json"
model_config_relative="configs/dfine/dfine_hgnetv2_l_coco.yml"

command = f"""
cd {dfine_path} && \
DFINE_PRED_DIR="{fold_output_dir}" \
python train.py \
  -c {model_config_relative} \
  --test-only \
  -r {fold_output_dir}/last.pth \
  -u val_dataloader.dataset.img_folder="{val_img_folder}" \
     val_dataloader.dataset.ann_file="{val_ann_file}" \
     remap_mscoco_category=False \
     num_classes=23 \
  --output-dir {fold_output_dir}
"""
!{command}

!ls -lah {fold_output_dir} | head -n 60


2026-02-16 10:50:57.999176: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-02-16 10:50:58.017587: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1771239058.039614   12404 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1771239058.046861   12404 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1771239058.065160   12404 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking 

In [21]:
import os, json, cv2
from collections import defaultdict

FOLD_DIR = "/content/drive/MyDrive/PCB_MC/Results/D-Fine/non_missing/fold_0"
PRED_JSON = os.path.join(FOLD_DIR, "predictions.json")
IMG_DIR   = "/content/drive/MyDrive/PCB_MC/Data/non_missing/kfold_data/fold_0/valid/images"
GT_JSON   = "/content/drive/MyDrive/PCB_MC/Data/non_missing/kfold_data/fold_0/valid/COCO_valid.json"

OUT_DIR = os.path.join(FOLD_DIR, "viz_dfine_tp_fp_fn")
os.makedirs(OUT_DIR, exist_ok=True)

# BGR
TP_C = (255, 0, 255)   # magenta
FP_C = (0, 165, 255)   # orange
FN_C = (0, 0, 255)     # red
WHITE = (255, 255, 255)
BLACK = (0, 0, 0)

CONF_TH = 0.30
IOU_TH  = 0.50

def xywh_to_xyxy(b):
    x, y, w, h = b
    return [x, y, x+w, y+h]

def iou(a, b):
    ax1, ay1, ax2, ay2 = a
    bx1, by1, bx2, by2 = b
    inter_x1 = max(ax1, bx1)
    inter_y1 = max(ay1, by1)
    inter_x2 = min(ax2, bx2)
    inter_y2 = min(ay2, by2)
    iw = max(0.0, inter_x2 - inter_x1)
    ih = max(0.0, inter_y2 - inter_y1)
    inter = iw * ih
    if inter == 0:
        return 0.0
    area_a = max(0.0, ax2-ax1) * max(0.0, ay2-ay1)
    area_b = max(0.0, bx2-bx1) * max(0.0, by2-by1)
    return inter / (area_a + area_b - inter + 1e-9)

def draw_label(img, text, x, y):
    cv2.putText(img, text, (x, y), cv2.FONT_HERSHEY_SIMPLEX, 0.5, BLACK, 3, cv2.LINE_AA)
    cv2.putText(img, text, (x, y), cv2.FONT_HERSHEY_SIMPLEX, 0.5, WHITE, 1, cv2.LINE_AA)

coco = json.load(open(GT_JSON, "r"))
id2file = {im["id"]: im["file_name"] for im in coco["images"]}

gt_by_image = defaultdict(list)
for ann in coco["annotations"]:
    gt_by_image[ann["image_id"]].append({
        "cls": int(ann["category_id"]),
        "bbox_xyxy": xywh_to_xyxy(ann["bbox"]),
    })

preds = json.load(open(PRED_JSON, "r"))
pred_by_image = defaultdict(list)
for p in preds:
    if float(p["score"]) < CONF_TH:
        continue
    pred_by_image[p["image_id"]].append({
        "cls": int(p["category_id"]),
        "score": float(p["score"]),
        "bbox_xyxy": xywh_to_xyxy(p["bbox"]),
    })

for image_id, fn in id2file.items():
    img_path = os.path.join(IMG_DIR, fn)
    img = cv2.imread(img_path)
    if img is None:
        continue

    gts = gt_by_image.get(image_id, [])
    prs = sorted(pred_by_image.get(image_id, []), key=lambda d: d["score"], reverse=True)

    gt_used = [False]*len(gts)
    pr_used = [False]*len(prs)

    # Match preds to GT (same class) by best IoU
    for pi, pr in enumerate(prs):
        best_iou, best_gi = 0.0, -1
        for gi, gt in enumerate(gts):
            if gt_used[gi]:
                continue
            if pr["cls"] != gt["cls"]:
                continue
            v = iou(pr["bbox_xyxy"], gt["bbox_xyxy"])
            if v > best_iou:
                best_iou, best_gi = v, gi
        if best_iou >= IOU_TH and best_gi >= 0:
            pr_used[pi] = True
            gt_used[best_gi] = True

    # Draw TPs and FPs
    for pi, pr in enumerate(prs):
        x1,y1,x2,y2 = map(int, pr["bbox_xyxy"])
        if pr_used[pi]:
            cv2.rectangle(img, (x1,y1), (x2,y2), TP_C, 2)
        else:
            cv2.rectangle(img, (x1,y1), (x2,y2), FP_C, 2)
            draw_label(img, f"FP {pr['cls']}:{pr['score']:.2f}", x1, max(15, y1-5))

    # Draw FNs (missed GT)
    for gi, gt in enumerate(gts):
        if not gt_used[gi]:
            x1,y1,x2,y2 = map(int, gt["bbox_xyxy"])
            cv2.rectangle(img, (x1,y1), (x2,y2), FN_C, 2)
            draw_label(img, f"FN {gt['cls']}", x1, max(15, y1-5))

    cv2.imwrite(os.path.join(OUT_DIR, fn), img)

print("✅ Saved TP/FP/FN overlays to:", OUT_DIR)


✅ Saved TP/FP/FN overlays to: /content/drive/MyDrive/PCB_MC/Results/D-Fine/non_missing/fold_0/viz_dfine_tp_fp_fn
